# Laya FP16 V2: does our optimization beat the compiled original?

Your T4 run reached **3.53×** at batch 1 / 128 tokens, comparing optimized compiled
inference with the original eager model. This notebook separates those two effects
and measures the **compiled complete prediction API**.

| Comparison | What it answers |
|---|---|
| Original eager → original compiled | How much comes from compilation? |
| Original eager → optimized eager | How much comes from the code changes alone? |
| Original compiled → optimized compiled | Do our changes help beyond compilation? |
| Legacy API → prepared compiled API | How fast is a complete application request? |

All paths use the unchanged **`vtava/Laya-1.2-speedup-standalone-fp16`** checkpoint.
No retraining, quantization, weight sharing or model upload. Candidates must preserve
every tested decision and stay within 0.01 probability drift. Failure is recorded;
the compiled original can win.

**Runtime → Change runtime type → GPU**, then **Run all**. Compilation across shapes
can take several minutes. The default examples are synthetic, not a held-out accuracy
benchmark. Upload representative cases below for a more meaningful application check.

In [ ]:
# Keep Colab's CUDA-enabled PyTorch; install the checkpoint's pinned dependencies.
%pip -q install "transformers==4.57.6" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json, subprocess, sys, tempfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select a GPU runtime first."
print("GPU:", torch.cuda.get_device_name(0), "PyTorch:", torch.__version__)
SOURCE_REVISION = "97ec99bbf0c3d093a10aa06ff0f8fceedd27495b"
MODEL_REVISION = "2891f7312d8e7c65b43a6e04c29cc226b06b4fac"
# An isolated checkout avoids changing any existing project.
REPO = Path(tempfile.mkdtemp(prefix="laya-v2-")) / "TinyCeNN-LM"
REPO.mkdir()
subprocess.run(["git", "init", "-q", str(REPO)], check=True)
subprocess.run(["git", "remote", "add", "origin", "https://github.com/vtavakkoli/TinyCeNN-LM.git"], cwd=REPO, check=True)
subprocess.run(["git", "fetch", "--depth", "1", "origin", SOURCE_REVISION], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO, check=True)
print("Pinned code:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

In [ ]:
# Both compiled controls receive identical settings.
LENGTHS = [128, 512]
BATCH_SIZES = [1, 4]
COMPILE_MODE = "reduce-overhead"  # Try "max-autotune" in a separate comparable run.
DYNAMIC_SHAPES = False          # Consider True for custom data with many distinct shapes.
API_MAX_LEN = 512
WARMUP = 5
ROUNDS = 7
REPEATS = 20
API_REPEATS = 5
MAX_PROBABILITY_DRIFT = 0.01
REPORT = Path("/content/laya_fp16_v2.json")
LOG = Path("/content/laya_fp16_v2.log")
CASES_JSONL = None

## Optional: your real inputs

Enable the upload below, or set `CASES_JSONL` to a file already in Colab. Each JSONL
line needs a `state` and optionally a `questions` mapping. For example:

```json
{"state":{"road_direction":"left","speed":7.0},"questions":{"steering":{"type":"choice","instructions":"Choose road direction.","criteria":["left","straight","right"]}}}
```

The complete API test uses these cases; the tensor-forward comparison keeps its
controlled synthetic lengths. Prepared inputs are checked against the original
sequence builder. The report records lengths and reference decision margins, not
verbatim states/questions. Include ambiguous and near-tied application examples.

In [ ]:
UPLOAD_CUSTOM_CASES = False
if UPLOAD_CUSTOM_CASES:
    from google.colab import files
    uploaded = files.upload()
    names = [name for name in uploaded if name.endswith(".jsonl")]
    assert len(names) == 1, "Upload exactly one .jsonl file."
    CASES_JSONL = Path("/content/laya_v2_custom_cases.jsonl")
    CASES_JSONL.write_bytes(uploaded[names[0]])
print("API input source:", "custom cases" if CASES_JSONL else "three synthetic example states")

In [ ]:
# Fullgraph=True is required for both compiled paths. A compile failure is not
# silently counted as a successful eager fallback. All timing begins after warmup.
command = [sys.executable, "-u", str(REPO / "scripts/benchmark_laya_fp16_v2.py"),
           "--device", "cuda", "--lengths", *map(str, LENGTHS),
           "--batch-sizes", *map(str, BATCH_SIZES), "--compile-mode", COMPILE_MODE,
           "--api-max-len", str(API_MAX_LEN), "--warmup", str(WARMUP),
           "--rounds", str(ROUNDS), "--repeats", str(REPEATS),
           "--api-repeats", str(API_REPEATS), "--tolerance", str(MAX_PROBABILITY_DRIFT),
           "--output", str(REPORT)]
if DYNAMIC_SHAPES: command += ["--dynamic"]
if CASES_JSONL: command += ["--cases-jsonl", str(CASES_JSONL)]
with LOG.open("w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code:
    raise RuntimeError(f"Benchmark stopped ({return_code}). Inspect {LOG}; partial JSON is retained. "
                       "For CUDA illegal-memory/device-assert errors, restart the runtime.")

In [ ]:
from IPython.display import Markdown, display
report = json.loads(REPORT.read_text())
def time_cell(rows, name):
    row = rows.get(name, {})
    return f"{row['median_ms']:.2f}" if row.get("passed") and "median_ms" in row else "FAILED / absent"
lines = ["| Batch | Tokens | Original eager ms | Original compiled ms | Optimized eager ms | Optimized compiled ms | Beyond compilation |",
         "|---:|---:|---:|---:|---:|---:|---:|"]
for item in report.get("forward", []):
    rows = item["candidates"]
    ratio = item["ratios"].get("optimization_beyond_compile")
    gain = f"{ratio:.3f}×" if ratio is not None else "unavailable"
    times = [time_cell(rows, name) for name in
             ("original_eager", "original_compiled", "optimized_eager", "optimized_compiled")]
    lines.append(f"| {item['batch_size']} | {item['sequence_length']} | " + " | ".join(times) + f" | {gain} |")
display(Markdown("\n".join(lines)))
print("Beyond-compilation ratio = original compiled time / optimized compiled time.")
print("Above 1: our changes help. Below 1: compiled original is faster.")

In [ ]:
api = report.get("end_to_end", {})
lines = ["| Complete prediction API | Mean request time, median round (ms) | Quality gate | Setup + validation + warmup (s) |",
         "|---|---:|---|---:|"]
for name, row in api.get("candidates", {}).items():
    timing = f"{row['median_ms']:.2f}" if "median_ms" in row else "—"
    lines.append(f"| {name} | {timing} | {'PASS' if row.get('passed') else 'FAIL'} | {row.get('setup_validation_warmup_seconds', 0):.2f} |")
display(Markdown("\n".join(lines)))
print("API ratios:", json.dumps(api.get("ratios", {}), indent=2))
print("Fastest accepted API:", api.get("winner"))
print("Case shapes and decision margins:", json.dumps(report.get("api_case_metadata", []), indent=2))
failures = []
for item in report.get("forward", []):
    failures += [(f"forward B={item['batch_size']} T={item['sequence_length']} / {name}", row)
                 for name, row in item["candidates"].items() if not row.get("passed")]
failures += [(f"API / {name}", row) for name, row in api.get("candidates", {}).items() if not row.get("passed")]
for name, row in failures:
    print("FAILED:", name, json.dumps(row, indent=2))
if not failures: print("All executed candidates passed their gates.")

## Interpret the result

Use the **compiled-original vs compiled-optimized ratio** for the benefit of our
implementation changes. Use **complete API timing** to choose an application runtime.
A large combined speedup alone does not demonstrate a new architecture advantage.

Timing order alternates across seven rounds; all round measurements are saved.
Close results can still be noise. An API request includes every question for one
state. Setup/compilation and one-time cached question preparation are excluded from
steady-state API timings. No checkpoint is published by this notebook.

Download the report and log below and send the JSON back for comparison.

In [ ]:
import zipfile
from google.colab import files
bundle = Path("/content/laya_fp16_v2_results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (REPORT, LOG):
        if path.exists(): archive.write(path, arcname=path.name)
files.download(str(bundle))